# 02 — Build Cross-Currency OIS Curves

USD-collateralized OIS discount curves for USD, EUR, GBP, JPY,
plus cross-currency basis (EUR/USD, GBP/USD, JPY/USD).
Synthetic market data. Downstream input for tensor PCA.

In [1]:
# Imports & settings
import ORE
import numpy as np
import scipy

today = ORE.Date(31, ORE.August, 2026)
ORE.Settings.instance().evaluationDate = today

calendar = ORE.TARGET()
settlement_days = 2
day_counter = ORE.Actual360()

print("ORE loaded. Eval date:", ORE.Settings.instance().evaluationDate)

ORE loaded. Eval date: August 31st, 2026


In [2]:
# Pillars
tenors = ["1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "15Y", "20Y", "30Y"]
periods = [ORE.Period(t) for t in tenors]

In [3]:
# Synthetic market data
ois_quotes = {
    "USD": [0.0400, 0.0395, 0.0390, 0.0385, 0.0385, 0.0390, 0.0400, 0.0405, 0.0410],
    "EUR": [0.0250, 0.0255, 0.0260, 0.0270, 0.0280, 0.0290, 0.0300, 0.0305, 0.0310],
    "GBP": [0.0450, 0.0445, 0.0440, 0.0435, 0.0435, 0.0440, 0.0445, 0.0450, 0.0455],
    "JPY": [0.0050, 0.0060, 0.0070, 0.0085, 0.0100, 0.0120, 0.0140, 0.0150, 0.0160],
}

xccy_basis_bps = {
    "EURUSD": [-15, -18, -20, -22, -24, -25, -26, -27, -28],
    "GBPUSD": [-8,  -9,  -10, -11, -12, -13, -14, -15, -16],
    "JPYUSD": [-30, -35, -40, -45, -50, -55, -58, -60, -62],
}

for ccy, q in ois_quotes.items():
    assert len(q) == len(tenors)
for pair, q in xccy_basis_bps.items():
    assert len(q) == len(tenors)

print("Synthetic data defined.")

Synthetic data defined.


In [4]:
# Quote handles
def make_quote_handles(values):
    quotes = [ORE.SimpleQuote(v) for v in values]
    return quotes, [ORE.QuoteHandle(q) for q in quotes]

ois_handles = {ccy: make_quote_handles(v)[1] for ccy, v in ois_quotes.items()}
basis_handles = {
    p: make_quote_handles([b / 10000.0 for b in bps])[1]
    for p, bps in xccy_basis_bps.items()
}
print("Quote handles created.")

Quote handles created.


In [5]:
# Built-in overnight indices per currency
overnight_indices = {
    "USD": ORE.Sofr(),
    "EUR": ORE.Estr(),
    "GBP": ORE.Sonia(),
    "JPY": ORE.Tona(),
}

for ccy, idx in overnight_indices.items():
    print(f"{ccy}: {idx.name()}  |  DC={idx.dayCounter()}  |  cal={idx.fixingCalendar()}")

USD: SOFRON Actual/360  |  DC=Actual/360 day counter  |  cal=SOFR fixing calendar calendar
EUR: ESTRON Actual/360  |  DC=Actual/360 day counter  |  cal=TARGET calendar
GBP: SoniaON Actual/365 (Fixed)  |  DC=Actual/365 (Fixed) day counter  |  cal=London stock exchange calendar
JPY: TonarON Actual/365 (Fixed)  |  DC=Actual/365 (Fixed) day counter  |  cal=Japan calendar


/tmp/ipykernel_27303/1196604616.py:6: FutureWarning: Tona is deprecated; use Tonar
  "JPY": ORE.Tona(),


In [6]:
# Build OIS rate helpers for a given currency
def build_ois_helpers(ccy):
    index = overnight_indices[ccy]
    helpers = []
    for period, quote in zip(periods, ois_handles[ccy]):
        helper = ORE.OISRateHelper(settlement_days, period, quote, index)
        helpers.append(helper)
    return helpers

# Quick check on USD
usd_helpers = build_ois_helpers("USD")
print(f"Built {len(usd_helpers)} USD OIS helpers")
for t, h in zip(tenors, usd_helpers):
    print(f"  {t}: maturity {h.maturityDate()}")

Built 9 USD OIS helpers
  1Y: maturity September 2nd, 2027
  2Y: maturity September 5th, 2028
  3Y: maturity September 4th, 2029
  5Y: maturity September 2nd, 2031
  7Y: maturity September 2nd, 2033
  10Y: maturity September 2nd, 2036
  15Y: maturity September 3rd, 2041
  20Y: maturity September 4th, 2046
  30Y: maturity September 5th, 2056


In [7]:
# Bootstrap all four OIS discount curves
ois_curves = {}
ois_curve_handles = {}

for ccy in overnight_indices:
    helpers = build_ois_helpers(ccy)
    curve = ORE.PiecewiseLogLinearDiscount(today, helpers, day_counter)
    curve.enableExtrapolation()
    ois_curves[ccy] = curve
    ois_curve_handles[ccy] = ORE.YieldTermStructureHandle(curve)
    print(f"{ccy} OIS curve bootstrapped. maxDate={curve.maxDate()}")

USD OIS curve bootstrapped. maxDate=September 5th, 2056
EUR OIS curve bootstrapped. maxDate=September 4th, 2056
GBP OIS curve bootstrapped. maxDate=September 4th, 2056
JPY OIS curve bootstrapped. maxDate=September 4th, 2056


In [16]:
# Link each overnight index to its bootstrapped forecast curve.
# The xccy helper forecasts the USD (base) leg, so USD SOFR needs a curve.
overnight_indices = {
    "USD": ORE.Sofr(ois_curve_handles["USD"]),
    "EUR": ORE.Estr(ois_curve_handles["EUR"]),
    "GBP": ORE.Sonia(ois_curve_handles["GBP"]),
    "JPY": ORE.Tona(ois_curve_handles["JPY"]),
}
print("Indices relinked to forecast curves.")

Indices relinked to forecast curves.


/tmp/ipykernel_27303/3708469058.py:7: FutureWarning: Tona is deprecated; use Tonar
  "JPY": ORE.Tona(ois_curve_handles["JPY"]),


In [17]:
# Sanity check: discount factors & zero rates
for ccy, curve in ois_curves.items():
    print(f"\n=== {ccy} ===")
    print(f"{'Tenor':>6} {'Maturity':>12} {'DF':>10} {'Zero%':>8}")
    for t, period in zip(tenors, periods):
        d = calendar.advance(today, period)
        df = curve.discount(d)
        zero = curve.zeroRate(d, day_counter, ORE.Continuous).rate()
        print(f"{t:>6} {str(d):>12} {df:>10.6f} {zero*100:>8.4f}")


=== USD ===
 Tenor     Maturity         DF    Zero%
    1Y August 31st, 2027   0.961025   3.9210
    2Y August 31st, 2028   0.924387   3.8721
    3Y August 31st, 2029   0.890139   3.8226
    5Y September 1st, 2031   0.825758   3.7725
    7Y August 31st, 2033   0.764882   3.7736
   10Y September 1st, 2036   0.677915   3.8299
   15Y September 2nd, 2041   0.548282   3.9472
   20Y August 31st, 2046   0.443417   4.0078
   30Y August 31st, 2056   0.289526   4.0721

=== EUR ===
 Tenor     Maturity         DF    Zero%
    1Y August 31st, 2027   0.975279   2.4688
    2Y August 31st, 2028   0.950162   2.5177
    3Y August 31st, 2029   0.924810   2.5675
    5Y September 1st, 2031   0.873340   2.6686
    7Y August 31st, 2033   0.821304   2.7716
   10Y September 1st, 2036   0.746778   2.8767
   15Y September 2nd, 2041   0.634798   2.9849
   20Y August 31st, 2046   0.539641   3.0399
   30Y August 31st, 2056   0.389573   3.0970

=== GBP ===
 Tenor     Maturity         DF    Zero%
    1Y August 31st,

In [18]:
# Build USD-collateralized foreign discount curves via xccy basis.
# Const-notional helper: pins the collateral (USD) curve and the two indices,
# then bootstraps the foreign (quote-currency) discount curve.

def build_xccy_basis_helpers(pair):
    fccy = pair[:3]                          # foreign currency, e.g. "EUR"
    base_index = overnight_indices["USD"]    # FX base currency = USD
    quote_index = overnight_indices[fccy]    # quote currency = foreign
    helpers = []
    for period, spread in zip(periods, basis_handles[pair]):
        h = ORE.ConstNotionalCrossCurrencyBasisSwapRateHelper(
            spread,                       # basis quote
            period,                       # tenor
            settlement_days,              # fixingDays
            calendar,                     # calendar
            ORE.ModifiedFollowing,        # convention
            False,                        # endOfMonth
            base_index,                   # baseCurrencyIndex (USD SOFR)
            quote_index,                  # quoteCurrencyIndex (foreign OIS)
            ois_curve_handles["USD"],     # collateralCurve (USD OIS)
            True,                         # isFxBaseCurrencyCollateralCurrency (USD)
            False,                        # isBasisOnFxBaseCurrencyLeg (basis on foreign)
            ORE.Quarterly,                # paymentFrequency (required for OIS legs)
            0,                            # paymentLag
        )
        helpers.append(h)
    return helpers

# Smoke test
eur_xccy_helpers = build_xccy_basis_helpers("EURUSD")
print(f"Built {len(eur_xccy_helpers)} EURUSD xccy helpers")
for t, h in zip(tenors, eur_xccy_helpers):
    print(f"  {t}: maturity {h.maturityDate()}")

Built 9 EURUSD xccy helpers
  1Y: maturity September 2nd, 2027
  2Y: maturity September 4th, 2028
  3Y: maturity September 3rd, 2029
  5Y: maturity September 2nd, 2031
  7Y: maturity September 2nd, 2033
  10Y: maturity September 2nd, 2036
  15Y: maturity September 2nd, 2041
  20Y: maturity September 3rd, 2046
  30Y: maturity September 4th, 2056


In [19]:
# Bootstrap USD-collateralized foreign discount curves
xccy_curves = {}
xccy_curve_handles = {}

for pair in xccy_basis_bps:
    fccy = pair[:3]
    helpers = build_xccy_basis_helpers(pair)
    curve = ORE.PiecewiseLogLinearDiscount(today, helpers, day_counter)
    curve.enableExtrapolation()
    xccy_curves[fccy] = curve
    xccy_curve_handles[fccy] = ORE.YieldTermStructureHandle(curve)
    print(f"{pair}: {fccy} USD-collateralized curve bootstrapped. maxDate={curve.maxDate()}")

EURUSD: EUR USD-collateralized curve bootstrapped. maxDate=September 4th, 2056
GBPUSD: GBP USD-collateralized curve bootstrapped. maxDate=September 4th, 2056
JPYUSD: JPY USD-collateralized curve bootstrapped. maxDate=September 4th, 2056


In [20]:
# Compare plain foreign OIS vs USD-collateralized foreign curve.
# The DF/zero difference reflects the cross-currency basis.
for fccy in xccy_curves:
    print(f"\n=== {fccy}: plain OIS vs USD-collateralized ===")
    print(f"{'Tenor':>6} {'DF_ois':>10} {'DF_xccy':>10} {'ΔZero(bps)':>11}")
    for t, period in zip(tenors, periods):
        d = calendar.advance(today, period)
        df_ois = ois_curves[fccy].discount(d)
        df_xccy = xccy_curves[fccy].discount(d)
        z_ois = ois_curves[fccy].zeroRate(d, day_counter, ORE.Continuous).rate()
        z_xccy = xccy_curves[fccy].zeroRate(d, day_counter, ORE.Continuous).rate()
        print(f"{t:>6} {df_ois:>10.6f} {df_xccy:>10.6f} {(z_xccy - z_ois)*1e4:>11.2f}")


=== EUR: plain OIS vs USD-collateralized ===
 Tenor     DF_ois    DF_xccy  ΔZero(bps)
    1Y   0.975279   0.976755      -14.91
    2Y   0.950162   0.953622      -17.90
    3Y   0.924810   0.930342      -19.59
    5Y   0.873340   0.882865      -21.37
    7Y   0.821304   0.835203      -23.63
   10Y   0.746778   0.765648      -24.59
   15Y   0.634798   0.660179      -25.75
   20Y   0.539641   0.569934      -26.92
   30Y   0.389573   0.424386      -28.12

=== GBP: plain OIS vs USD-collateralized ===
 Tenor     DF_ois    DF_xccy  ΔZero(bps)
    1Y   0.956938   0.957695       -7.80
    2Y   0.916513   0.918151       -8.79
    3Y   0.878776   0.881309       -9.45
    5Y   0.808235   0.812429      -10.20
    7Y   0.742263   0.748316      -11.44
   10Y   0.649449   0.657697      -12.43
   15Y   0.518789   0.529697      -13.67
   20Y   0.411634   0.424293      -14.93
   30Y   0.258633   0.271756      -16.26

=== JPY: plain OIS vs USD-collateralized ===
 Tenor     DF_ois    DF_xccy  ΔZero(bps)
 

In [21]:
# Refactor the entire curve build into one reusable function.
# Given OIS quotes and xccy basis, returns bootstrapped curves.

def build_curves(ois_q, basis_bps):
    # Quote handles
    ois_h = {ccy: [ORE.QuoteHandle(ORE.SimpleQuote(v)) for v in vals]
             for ccy, vals in ois_q.items()}
    basis_h = {p: [ORE.QuoteHandle(ORE.SimpleQuote(b / 10000.0)) for b in bps]
               for p, bps in basis_bps.items()}

    # Overnight indices (unlinked first pass)
    idx = {
        "USD": ORE.Sofr(),
        "EUR": ORE.Estr(),
        "GBP": ORE.Sonia(),
        "JPY": ORE.Tona(),
    }

    # Bootstrap OIS discount curves
    ois_c, ois_ch = {}, {}
    for ccy in idx:
        helpers = [
            ORE.OISRateHelper(settlement_days, period, q, idx[ccy])
            for period, q in zip(periods, ois_h[ccy])
        ]
        c = ORE.PiecewiseLogLinearDiscount(today, helpers, day_counter)
        c.enableExtrapolation()
        ois_c[ccy] = c
        ois_ch[ccy] = ORE.YieldTermStructureHandle(c)

    # Relink indices to their forecast curves
    idx = {
        "USD": ORE.Sofr(ois_ch["USD"]),
        "EUR": ORE.Estr(ois_ch["EUR"]),
        "GBP": ORE.Sonia(ois_ch["GBP"]),
        "JPY": ORE.Tona(ois_ch["JPY"]),
    }

    # Bootstrap USD-collateralized foreign discount curves
    xccy_c, xccy_ch = {}, {}
    for pair in basis_bps:
        fccy = pair[:3]
        helpers = []
        for period, spread in zip(periods, basis_h[pair]):
            h = ORE.ConstNotionalCrossCurrencyBasisSwapRateHelper(
                spread, period, settlement_days, calendar,
                ORE.ModifiedFollowing, False,
                idx["USD"], idx[fccy], ois_ch["USD"],
                True, False, ORE.Quarterly, 0,
            )
            helpers.append(h)
        c = ORE.PiecewiseLogLinearDiscount(today, helpers, day_counter)
        c.enableExtrapolation()
        xccy_c[fccy] = c
        xccy_ch[fccy] = ORE.YieldTermStructureHandle(c)

    return ois_c, xccy_c

# Verify the refactor matches our earlier build
_ois, _xccy = build_curves(ois_quotes, xccy_basis_bps)
print("Refactored build OK. Curves:", list(_ois), "| xccy:", list(_xccy))

Refactored build OK. Curves: ['USD', 'EUR', 'GBP', 'JPY'] | xccy: ['EUR', 'GBP', 'JPY']


/tmp/ipykernel_27303/3315689312.py:16: FutureWarning: Tona is deprecated; use Tonar
  "JPY": ORE.Tona(),
/tmp/ipykernel_27303/3315689312.py:36: FutureWarning: Tona is deprecated; use Tonar
  "JPY": ORE.Tona(ois_ch["JPY"]),


In [22]:
# Extract continuously-compounded zero rates at the pillar tenors.
# Order: USD OIS, then each xccy-adjusted foreign discount curve.
curve_order = ["USD", "EUR", "GBP", "JPY"]   # USD from OIS; EUR/GBP/JPY from xccy

def extract_zero_rates(ois_c, xccy_c):
    rows = []
    for ccy in curve_order:
        curve = ois_c["USD"] if ccy == "USD" else xccy_c[ccy]
        row = [
            curve.zeroRate(calendar.advance(today, p), day_counter,
                           ORE.Continuous).rate()
            for p in periods
        ]
        rows.append(row)
    return np.array(rows)  # shape (currencies, tenors)

base_zeros = extract_zero_rates(_ois, _xccy)
print("Zero-rate matrix shape:", base_zeros.shape)  # (4, 9)
print(np.round(base_zeros * 100, 4))

Zero-rate matrix shape: (4, 9)
[[3.921  3.8721 3.8226 3.7725 3.7736 3.8299 3.9472 4.0078 4.0721]
 [2.3197 2.3387 2.3716 2.4548 2.5354 2.6309 2.7274 2.7708 2.8158]
 [4.2633 4.2054 4.1501 4.0931 4.0819 4.1282 4.1737 4.225  4.2802]
 [0.1963 0.245  0.2975 0.3991 0.4976 0.6509 0.8265 0.9104 0.9957]]


In [25]:
# Generate N scenarios using a low-rank FACTOR model so PCA/Tensor-PCA
# have real structure to find: shared level/slope/curvature across tenors,
# correlated across currencies, plus a smaller basis-divergence factor.

rng = np.random.default_rng(42)
n_scenarios = 200

t = np.linspace(0.0, 1.0, len(tenors))           # normalized tenor axis
level = np.ones_like(t)                            # flat
slope = t - t.mean()                               # increasing
curv  = (t - 0.5) ** 2; curv -= curv.mean()        # U-shape

# Currency loadings (common move + FX-basis divergence)
ccy_common  = np.array([1.0, 0.9, 0.85, 0.7])      # USD, EUR, GBP, JPY
ccy_diverge = np.array([0.0, 1.0, 0.6, 1.4])       # foreign-only basis wobble

# Factor volatilities (rate units)
sd_level, sd_slope, sd_curv, sd_basis = 0.0010, 0.0006, 0.0003, 0.0004

tensor = np.empty((n_scenarios, len(curve_order), len(tenors)))

for s in range(n_scenarios):
    f_level = rng.normal(0, sd_level)
    f_slope = rng.normal(0, sd_slope)
    f_curv  = rng.normal(0, sd_curv)
    shared  = f_level * level + f_slope * slope + f_curv * curv   # (n_tenors,)

    f_basis = rng.normal(0, sd_basis)

    dev = (np.outer(ccy_common, shared)
           + np.outer(ccy_diverge, f_basis * level))              # (n_ccy, n_tenors)

    tensor[s] = base_zeros + dev

print("Tensor shape (scenarios, currencies, tenors):", tensor.shape)
print("Std zero (bps):\n", np.round(tensor.std(axis=0) * 1e4, 2))

Tensor shape (scenarios, currencies, tenors): (200, 4, 9)
Std zero (bps):
 [[10.32 10.15 10.05  9.99  9.99 10.03 10.12 10.26 10.46]
 [10.36 10.19 10.07 10.01  9.99 10.03 10.11 10.24 10.42]
 [ 9.27  9.12  9.01  8.96  8.94  8.98  9.05  9.18  9.35]
 [ 9.34  9.21  9.11  9.06  9.04  9.06  9.12  9.22  9.36]]


In [26]:
# Save tensor + axis labels for notebook 03 (tensor PCA)
import os
out_dir = os.path.join("..", "data")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "zero_rate_tensor.npz")

np.savez(
    out_path,
    tensor=tensor,
    currencies=np.array(curve_order),
    tenors=np.array(tenors),
)
print("Saved:", os.path.abspath(out_path))
print("  tensor", tensor.shape, "| currencies", curve_order, "| tenors", tenors)

Saved: /home/jonty/repos/ore/Examples/xccy-tensor-pca/data/zero_rate_tensor.npz
  tensor (200, 4, 9) | currencies ['USD', 'EUR', 'GBP', 'JPY'] | tenors ['1Y', '2Y', '3Y', '5Y', '7Y', '10Y', '15Y', '20Y', '30Y']
